<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px;">
<img src="../Modulo III/Figures/iteso.jpg" style="height:110px; background:white; padding:6px; border-radius:8px;"/>
<div style="border-left:2px solid white; padding-left:28px; color:white;">
<h1 style="font-size:1.5em;">Ingeniería y Ciencia de Datos</h1>
<h3>Laboratorio de Procesamiento de Datos</h3>
<p>Módulo III: Selección de características: varianza y correlación</p>
</div></div>


# Introducción

En este archivo de jupyter estudiaremos **dos filtros descriptivos** para revisar columnas antes de un análisis: la varianza y la correlación. 

- Un filtro calcula una característica de los datos y ayuda a detectar columnas constantes o redundantes; no demuestra que una variable sea importante ni que mejore una predicción.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import VarianceThreshold

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})

DATA_DIR = 'Data'
FIGURES_DIR = os.path.join(DATA_DIR, 'Figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


In [ ]:
def mostrar_figura(nombre):
    archivo_figura = os.path.join(FIGURES_DIR, f'{nombre}.png')
    plt.show()
    plt.close()

<a id="varianza"></a>
## 1. Selección de características por varianza

Una característica con mayor varianza significa que el valor dentro de esa característica varía o tiene una cardinalidad alta. Por otro lado, una menor varianza significa que el valor dentro de la característica es similar, y una varianza cero significa que la característica tiene el mismo valor.

Intuitivamente, queremos tener una característica variada, ya que no deseamos que nuestro modelo predictivo esté sesgado. Por eso, podríamos seleccionar la característica basándonos en la varianza que seleccionamos previamente.

La selección de características por varianza es un método sencillo para eliminar características en función de la varianza esperada dentro de cada una. Sin embargo, este método presenta algunas desventajas. La selección de características mediante umbral de varianza solo considera las características de entrada (X) sin tener en cuenta la información de la variable dependiente (y). Por lo tanto, solo resulta útil para eliminar características en modelos no supervisados, no en modelos supervisados.

La **varianza** mide cuánto se dispersan los valores de una columna respecto de su media. Si casi todos los registros tienen el mismo valor, esa columna distingue poco entre observaciones. Un filtro de varianza detecta columnas constantes o de dispersión muy pequeña, **no consulta una respuesta ni demuestra utilidad para una tarea**.

Para una característica $x$ con $n$ observaciones, usaremos la varianza poblacional:

$$\operatorname{Var}(x)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2,\qquad \bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i.$$

Es la convención de `VarianceThreshold` y coincide con `pandas.Series.var(ddof=0)`. Si se usa `ddof=1`, se calcula una varianza muestral con denominador $n-1$ y se obtiene un valor ligeramente distinto.

#### Procedimiento paso a paso:

El proceso de uso del método del umbral de varianza comprende los siguientes pasos:

1. Calcula la varianza para cada característica del conjunto de datos.
2. Compare cada variación con el umbral predefinido.
3. Descartar las características con una varianza inferior al umbral.
4. Conservar las características con suficiente variabilidad.

Esta técnica no es supervisada , lo que significa que no tiene en cuenta las etiquetas objetivo al seleccionar las características. Es más eficaz como filtro inicial antes de aplicar métodos más complejos.

### Cómo elegir un umbral

`VarianceThreshold(t)` conserva las columnas cuya varianza es **estrictamente mayor** que $t$. El umbral 0 elimina constantes, cualquier umbral mayor también puede eliminar indicadores poco frecuentes. Reporten el umbral, las columnas eliminadas y las conservadas. Si todas las columnas quedan debajo del umbral, el filtro no tiene características que retener y se debe revisar la decisión.

La escala importa: multiplicar una medida por 1,000 multiplica su varianza por $1{,}000{,}000$. Por eso un umbral absoluto solo tiene sentido en unidades conocidas. 

Estandarizar antes de filtrar suele llevar las columnas no constantes a varianza 1 y oculta cuáles tenían poca variación original. 


In [ ]:
binarias = pd.DataFrame({'constante': np.ones(100), 'rara': [1] + [0] * 99,
                         'balanceada': [0, 1] * 50, 'moderada': [1] * 10 + [0] * 90})
binarias

In [ ]:
# Obtener la varianza
binarias.var(ddof=0).rename('Varianza')

In [ ]:
#help(VarianceThreshold)

In [ ]:
# Elegimos umbrales para aplicar el filtro y aplicamos VarianceThreshold


### Ejemplo: mediciones de tumores

**Breast Cancer Wisconsin** contiene 30 mediciones numéricas de muestras celulares de tumores. Aquí **no usaremos la etiqueta de diagnóstico ni entrenaremos un clasificador**. Añadimos una columna constante y una copia de `mean radius` para ilustrar un filtro por varianza.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
tumores = cancer.data.copy()
tumores.info()

In [ ]:
# Agregamos una nueva variable
tumores['constante'] = 1
tumores['copia_radio'] = tumores['mean radius']

#Obtenemos la varianza
varianzas_tumores = tumores.var(ddof=0).sort_values()
varianzas_tumores.head(8).rename('Varianza poblacional').to_frame()

In [ ]:
tumores

In [ ]:
#Aplicamos el filtro por la varianza


In [ ]:
#Obtener el filtro

In [ ]:
#Columnas filtradas


In [ ]:
#Columnas eliminadas por varianza


## Casos de uso y aplicaciones

- Clasificación de texto: Los datos dispersos de alta dimensión (por ejemplo, vectores TF-IDF ) suelen contener muchas características con varianza nula o baja.
- Bioinformática: Los datos genómicos contienen miles de niveles de expresión génica, muchos de los cuales varían poco.
- Redes de sensores: Las lecturas de los sensores pueden permanecer constantes durante largos períodos, lo que introduce características de baja varianza.
- En todos estos ámbitos, la aplicación del umbral de varianza puede reducir el ruido y el tiempo de cálculo.

#### Ventajas y limitaciones

##### Ventajas
- Sencillez: Muy fácil de entender e implementar.
- Velocidad: Computacionalmente eficiente incluso con grandes conjuntos de datos.
- Utilidad de preprocesamiento: Útil como primer paso en el proceso de selección de características.

##### Limitaciones
- Ignora la variable objetivo: No puede evaluar la relevancia de una característica con respecto al resultado.
- No es eficaz para todas las características de baja varianza: algunas características de baja varianza aún podrían ser importantes para la clasificación .
- Falla con características redundantes: No puede detectar la multicolinealidad ni las características correlacionadas con alta varianza.

<a id="correlacion"></a>
## 2. Selección de características por correlación

La **correlación** resume la intensidad y dirección de una asociación entre dos columnas. Se usa para detectar posibles redundancias: si dos variables numéricas contienen información muy parecida, el equipo puede decidir conservar una por razones de interpretación, costo o facilidad de medición.

La correlación de **Pearson** mide asociación **lineal**:

$$r_{xy}=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_i(x_i-\bar x)^2\sum_i(y_i-\bar y)^2}}.$$

Su valor está entre -1 y 1. Un valor cercano a 1 indica asociación lineal positiva, cerca de -1, negativa, cerca de 0, poca asociación **lineal** en los datos observados. 

**Spearman** calcula Pearson sobre los rangos. Detecta una relación monótona aunque no sea lineal y reduce la influencia de la escala de valores extremos. No detecta cualquier dependencia: si la asociación tiene forma de U, puede resultar cercana a cero.

### Regla de filtrado por pares

Una estrategia sencilla calcula $|r|$ para cada par y, si supera un umbral como 0.85 o 0.90, elimina una de las dos columnas. La función de abajo usa un recorrido voraz y el orden de las columnas: conserva la primera y descarta las siguientes que excedan el umbral frente a alguna ya retenida. Por tanto, cambiar el orden puede cambiar qué variable se conserva.

El umbral no es una ley natural. Correlación alta sugiere redundancia por pares, pero no garantiza equivalencia, correlación baja no prueba independencia. Una constante tiene correlación indefinida, así que debe identificarse antes.

### Ejemplo conceptual: correlación cero con dependencia

Para $x=(-2,-1,0,1,2)$ y $z=x^2=(4,1,0,1,4)$, Pearson es cero: los productos positivos y negativos se cancelan por simetría. Sin embargo, al conocer x se conoce z exactamente. 

En la práctica con mediciones celulares, añadimos deliberadamente una copia exacta de `mean radius`. Esperamos que el filtro retire una de esas dos columnas, pero **no entrenaremos modelos** ni usaremos clases diagnósticas para decidir. La actividad es exploratoria.

In [ ]:
#Dataset ya después de aplicarle el filtro de varianzas tumores_var


In [ ]:
# Obtenemos la correlación de pearson


In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(6,4)
sns.heatmap(matriz)

In [ ]:
#Función para seleccionar las características correlacionada
def seleccionar_correlacion(datos, umbral=0.90, metodo='pearson'):
    matriz = datos.corr(method=metodo).abs()
    conservadas = []
    for columna in datos.columns:
        if not conservadas or not (matriz.loc[columna, conservadas] > umbral).any():
            conservadas.append(columna)
    return conservadas

In [ ]:
# columnas correlacionadas con umbral


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
primeras = list(tumores_var.columns[:9]) + ['copia_radio']
for eje, metodo in zip(ejes, ['pearson', 'spearman']):
    matriz = tumores_var[primeras].corr(method=metodo)
    sns.heatmap(matriz, vmin=-1, vmax=1, center=0, cmap='RdBu_r', annot=True, fmt='.1f', ax=eje)
    eje.set_title(f'Correlación {metodo}: mediciones de entrenamiento')
fig.tight_layout()
mostrar_figura('04_correlaciones')

In [ ]:
comparacion_filtros = pd.DataFrame([
    {'etapa': 'Características iniciales', 'columnas': tumores.shape[1],
     'columnas retiradas en esta etapa': 0},
    {'etapa': 'Después de retirar varianza cero', 'columnas': tumores_var.shape[1],
     'columnas retiradas en esta etapa': tumores.shape[1] - tumores_var.shape[1]},
    {'etapa': 'Después del filtro Pearson |r| > 0.90', 'columnas': tumores_cor.shape[1],
     'columnas retiradas en esta etapa': tumores_var.shape[1] - tumores_cor.shape[1]},
])
comparacion_filtros

# Práctica de laboratorio: variabilidad y relaciones en vinos tintos


In [ ]:
CARPETA_VINO = os.path.join('Data')
ARCHIVO_VINO = os.path.join(CARPETA_VINO, 'winequality-red.csv')

vino = pd.read_csv(ARCHIVO_VINO, sep=';')
vino.info()

In [ ]:
print('Dimensiones:', vino.shape)
print('Columnas:', vino.columns.tolist())


In [ ]:
print('Celdas faltantes:', int(vino.isna().sum().sum()))
print('Filas duplicadas:', int(vino.duplicated().sum()))
display(vino['quality'].value_counts().sort_index().rename_axis('quality').to_frame('muestras'))


####  Varianza de las mediciones

Separaremos las 11 mediciones fisicoquímicas de `quality`. Primero observaremos varianza y escala de cada característica. Luego compararemos tres umbrales sobre los valores originales. El umbral 0 busca constantes; 0.01 y 0.1 son ejemplos para explorar, **no recomendaciones universales**.

La diferencia de magnitudes entre columnas importa: comparar directamente la varianza de `density` con la de `total sulfur dioxide` puede reflejar sobre todo sus unidades. Por eso, esta tabla sirve para inspeccionar y practicar, no para declarar irrelevante una medida basándose únicamente en una varianza pequeña.

Al final estandarizaremos solo para ilustrar el efecto en las varianzas. Si todas las características se centran y escalan, `VarianceThreshold` deja de distinguir su variabilidad original. No usaremos esta estandarización para entrenar un modelo.

In [ ]:
from sklearn.preprocessing import StandardScaler

objetivo_vino = 'quality'
caracteristicas_vino = vino.drop(columns=[objetivo_vino])
varianzas_vino = caracteristicas_vino.var(ddof=0).sort_values()

resumen_varianzas_vino = pd.DataFrame({
    'varianza_poblacional': varianzas_vino,
    'desviacion_estandar': caracteristicas_vino.std(ddof=0).reindex(varianzas_vino.index),
    'minimo': caracteristicas_vino.min().reindex(varianzas_vino.index),
    'maximo': caracteristicas_vino.max().reindex(varianzas_vino.index)
})
resumen_varianzas_vino.round(5)


In [ ]:
for umbral_vino in [0, 0.01, 0.10]:
    filtro_vino = VarianceThreshold(threshold=umbral_vino).fit(caracteristicas_vino)
    retenidas = caracteristicas_vino.columns[filtro_vino.get_support()].tolist()
    eliminadas = caracteristicas_vino.columns[~filtro_vino.get_support()].tolist()
    print(f'Umbral {umbral_vino}: conserva {len(retenidas)}; elimina {eliminadas}')



In [ ]:
escalador_vino = StandardScaler().fit(caracteristicas_vino)
vino_estandarizado = pd.DataFrame(escalador_vino.transform(caracteristicas_vino),
                                  columns=caracteristicas_vino.columns)

vino_estandarizado.var(ddof=0).rename('varianza_tras_estandarizar').to_frame().round(5)

In [ ]:
fig, eje = plt.subplots(figsize=(10, 5))
varianzas_vino.sort_values().plot.barh(ax=eje, color='#245b8f')
eje.set_xscale('log')
eje.set(title='Varianza de características fisicoquímicas (escala logarítmica)',
       xlabel='Varianza poblacional, unidades originales', ylabel='Característica')
mostrar_figura('06_varianza_vino_uci')

#### Correlación entre mediciones del vino

Ahora calcularemos Pearson y Spearman únicamente entre las once variables fisicoquímicas. La puntuación `quality` no se incluye en el filtro de características. Después la compararemos descriptivamente con algunas mediciones para practicar la lectura de asociaciones, **sin entrenar un modelo**.

La selección de pares con $|r|>0.85$ es una regla de demostración. Revisen la matriz antes de retirar una variable: una asociación alta puede señalar medidas redundantes, pero se conserva la que tenga más sentido para la pregunta, interpretación o calidad de medición. Cambiar el umbral puede cambiar cuántas columnas quedan.

In [ ]:
correlaciones_pearson_vino = caracteristicas_vino.corr(method='pearson')
correlaciones_spearman_vino = caracteristicas_vino.corr(method='spearman')
fig, ejes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(correlaciones_pearson_vino, vmin=-1, vmax=1, center=0, cmap='RdBu_r', ax=ejes[0])
ejes[0].set_title('Pearson: asociación lineal')
sns.heatmap(correlaciones_spearman_vino, vmin=-1, vmax=1, center=0, cmap='RdBu_r', ax=ejes[1])
ejes[1].set_title('Spearman: asociación monótona por rangos')
fig.tight_layout()
mostrar_figura('07_correlacion_vino_uci')


In [ ]:
mascara_superior = np.triu(np.ones(correlaciones_pearson_vino.shape, dtype=bool), k=1)
pares_pearson_vino = correlaciones_pearson_vino.abs().where(mascara_superior).stack().sort_values(ascending=False)
pares_pearson_vino.index.names = ['variable_1', 'variable_2']
print('Diez pares con mayor |Pearson| entre características:')
pares_pearson_vino.head(10).rename('|Pearson|').to_frame()


In [ ]:
tabla_umbral_vino = []
for umbral in [0.85, 0.90, 0.95]:
    retenidas = seleccionar_correlacion(caracteristicas_vino, umbral=umbral, metodo='pearson')
    tabla_umbral_vino.append({'umbral |Pearson|': umbral, 'columnas retenidas': len(retenidas),
                              'columnas excluidas': len(caracteristicas_vino.columns) - len(retenidas),
                              'excluidas': ', '.join(columna for columna in caracteristicas_vino if columna not in retenidas)})
pd.DataFrame(tabla_umbral_vino)

relacion_quality = vino.corr(method='spearman')['quality'].drop('quality').sort_values()
relacion_quality.rename('Spearman descriptivo con quality').to_frame()